# Material Dataset Study Replotting

Use this notebook after running `training_scripts/run_material_dataset_study.py`. It reloads the saved metric tables and recreates editable summary plots for localization, false positives, and false negatives.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Point this at a completed run_material_dataset_study.py output directory.
OUTPUT_DIR = Path("../outputs/material_dataset_study")
METRICS_CSV = OUTPUT_DIR / "dataset_study_metrics.csv"
CONFIG_JSON = OUTPUT_DIR / "material_dataset_study_config.json"

CASE_LABELS = {
    "random_min_distance": "Blob, min distance",
    "tight_random_spacing": "Blob, tight spacing",
    "sto_ase": "STO, ASE cubic",
    "ws2_mx2": "WS2, ASE mx2 hex",
}

CASE_ORDER = list(CASE_LABELS)
PALETTE = {
    "random_min_distance": "#4E79A7",
    "tight_random_spacing": "#F28E2B",
    "sto_ase": "#59A14F",
    "ws2_mx2": "#E15759",
}

# Global style knobs for final figures.
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
})

In [ ]:
metrics = pd.read_csv(METRICS_CSV)
metrics["model_label"] = metrics["model"].map(CASE_LABELS).fillna(metrics["model"])
metrics["test_label"] = metrics["test_case"].map(CASE_LABELS).fillna(metrics["test_case"])
metrics["fp_per_image"] = metrics["fp"] / metrics["samples"].clip(lower=1)
metrics["fn_per_image"] = metrics["fn"] / metrics["samples"].clip(lower=1)
metrics["tp_per_image"] = metrics["tp"] / metrics["samples"].clip(lower=1)

metrics

In [ ]:
if CONFIG_JSON.exists():
    config = json.loads(CONFIG_JSON.read_text())
    dataset_summaries = pd.DataFrame([
        {
            "case": name,
            "label": payload["label"],
            "class": payload["class"],
            "pixel_size_angstrom": payload["summary"]["pixel_size_angstrom"],
            "feature_sigma_px": payload["summary"]["feature_sigma_px"],
            "feature_sigma_angstrom": payload["summary"]["feature_sigma_angstrom"],
            "median_nn_px": payload["summary"]["median_nn_px"],
            "median_nn_angstrom": payload["summary"]["median_nn_angstrom"],
            "visible_points_in_example": payload["summary"]["visible_points_in_example"],
        }
        for name, payload in config["datasets"].items()
    ])
else:
    dataset_summaries = pd.DataFrame()

dataset_summaries

In [ ]:
def metric_matrix(df, value, model_order=CASE_ORDER, test_order=CASE_ORDER):
    table = df.pivot(index="model", columns="test_case", values=value)
    return table.reindex(index=model_order, columns=test_order)


def plot_matrix(value, title, cmap="viridis", vmin=None, vmax=None, fmt=".3f", colorbar_label=None):
    matrix = metric_matrix(metrics, value)
    fig, ax = plt.subplots(figsize=(7.0, 5.8), constrained_layout=True)
    image = ax.imshow(matrix.values.astype(float), cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("Held-out test dataset")
    ax.set_ylabel("Training dataset")
    ax.set_xticks(np.arange(len(CASE_ORDER)), [CASE_LABELS[c] for c in CASE_ORDER], rotation=25, ha="right")
    ax.set_yticks(np.arange(len(CASE_ORDER)), [CASE_LABELS[c] for c in CASE_ORDER])
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value_here = matrix.values[row, col]
            text = "NA" if pd.isna(value_here) else format(float(value_here), fmt)
            ax.text(col, row, text, ha="center", va="center", color="white", fontsize=9)
    cbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    if colorbar_label:
        cbar.set_label(colorbar_label)
    return fig, ax


plot_matrix("f1", "F1 score", cmap="YlGnBu", vmin=0, vmax=1, fmt=".3f");
plot_matrix("rmse", "Localization RMSE", cmap="magma_r", vmin=0, fmt=".2f", colorbar_label="pixels");

In [ ]:
plot_matrix("fp_per_image", "False positives per image", cmap="magma", vmin=0, fmt=".1f");
plot_matrix("fn_per_image", "False negatives per image", cmap="magma", vmin=0, fmt=".1f");

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True, sharey=False)
x = np.arange(len(CASE_ORDER))
width = 0.8 / len(CASE_ORDER)

for test_index, test_case in enumerate(CASE_ORDER):
    subset = metrics[metrics["test_case"] == test_case].set_index("model").reindex(CASE_ORDER)
    offset = (test_index - (len(CASE_ORDER) - 1) / 2) * width
    axes[0].bar(x + offset, subset["fp_per_image"], width=width, label=CASE_LABELS[test_case], color=PALETTE[test_case])
    axes[1].bar(x + offset, subset["fn_per_image"], width=width, label=CASE_LABELS[test_case], color=PALETTE[test_case])

for ax, title, ylabel in [
    (axes[0], "False positives", "FP per image"),
    (axes[1], "False negatives", "FN per image"),
]:
    ax.set_title(title)
    ax.set_xticks(x, [CASE_LABELS[c] for c in CASE_ORDER], rotation=25, ha="right")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)

axes[1].legend(title="Test dataset", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("FP/FN rates by training and held-out dataset")
fig

In [ ]:
diagonal = metrics[metrics["model"] == metrics["test_case"]].set_index("model").reindex(CASE_ORDER)

fig, ax = plt.subplots(figsize=(8.0, 4.6), constrained_layout=True)
x = np.arange(len(CASE_ORDER))
ax.bar(x - 0.18, diagonal["fp_per_image"], width=0.36, label="False positives", color="#FF3B30")
ax.bar(x + 0.18, diagonal["fn_per_image"], width=0.36, label="False negatives", color="#1E8BFF")
ax.set_xticks(x, [CASE_LABELS[c] for c in CASE_ORDER], rotation=25, ha="right")
ax.set_ylabel("Count per image")
ax.set_title("In-distribution FP/FN rates")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig

In [ ]:
# Save the current matplotlib figure. Change the path/name as needed.
# fig.savefig(OUTPUT_DIR / "paper_fp_fn_rates.png", bbox_inches="tight", dpi=600)